In [48]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [49]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform= v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True)
    ])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=False,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True)
    ])
)

In [50]:
print(f"number of training example: {len(training_data)}")
print(f"number of test example: {len(test_data)}")

number of training example: 60000
number of test example: 10000


In [51]:
train_dataloader = DataLoader(
    training_data,
    batch_size=64,
    shuffle=True
)

test_dataloader = DataLoader(
    test_data,
    batch_size=64,
    shuffle=True
)

In [52]:
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(28*28, 512),
        nn.ReLU(),
        nn.Linear(512,512),
        nn.ReLU(),
        nn.Linear(512,10)
    )


  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits



In [65]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else 'cpu'
print(device)

cuda


In [66]:
model = NeuralNetwork().to(device)

In [67]:
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [68]:
# hyperparameters

learning_rate = 1e-3
batch_size = 64
epochs = 100

#loss function
loss_fn = nn.CrossEntropyLoss()
#optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [69]:
print(len(train_dataloader.dataset))

60000


In [70]:
examples, labels = next(iter(train_dataloader))
print(f"shape of example: {examples.shape}")
print(f"shape of labels: {labels.shape}")

shape of example: torch.Size([64, 1, 28, 28])
shape of labels: torch.Size([64])


In [76]:
def train_loop(dataloader, model, loss_fn, optimizer):
  size = len(dataloader.dataset)

  # set the model to training mode: important for batch
  # normalization and dropout layers
  # adding this is the bes practice

  model.train()

  for batch, (X, y) in enumerate(dataloader):
    #add the tensors to the GPU
    X, y = X.to(device), y.to(device)

    #compute prediction and loss
    pred = model(X)
    loss = loss_fn(pred, y)

    #backpropagation
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if batch % 100 == 0:
      loss, current = loss.item(), batch * batch_size + len(X)
      print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")



In [77]:
def test_loop(dataloader, model, loss_fn):
  # set the model to evaluation mode: important for batch
  # normalization and dropout layers
  # adding this is the bes practice
  model.eval()

  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  test_loss, correct = 0, 0

  # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
  # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True

  with torch.no_grad():
    for X, y in dataloader:
      #add the tensors to the GPU
      X, y = X.to(device), y.to(device)

      pred = model(X)
      test_loss += loss_fn(pred, y).item()
      correct += (pred.argmax(1) == y).type(torch.float).sum().item()

  test_loss /= num_batches
  correct /=size

  print(f"Test Error: \n Accuracy: {(100 * correct):> 0.1f}%, Avg loss: {test_loss:>8f} \n")

In [78]:
for epoch in range(epochs):
  print(f"Epoch {epoch+1}\n--------------------------------------")
  train_loop(train_dataloader, model, loss_fn, optimizer)
  test_loop(test_dataloader, model, loss_fn)

print("Done!")

Epoch 1
--------------------------------------
loss: 2.300941 [   64/60000]
loss: 2.278692 [ 6464/60000]
loss: 2.270698 [12864/60000]
loss: 2.256342 [19264/60000]
loss: 2.253022 [25664/60000]
loss: 2.211791 [32064/60000]
loss: 2.192112 [38464/60000]
loss: 2.190135 [44864/60000]
loss: 2.176744 [51264/60000]
loss: 2.161208 [57664/60000]
Test Error: 
 Accuracy:  42.1%, Avg loss: 2.146242 

Epoch 2
--------------------------------------
loss: 2.188542 [   64/60000]
loss: 2.120216 [ 6464/60000]
loss: 2.132495 [12864/60000]
loss: 2.077343 [19264/60000]
loss: 2.034745 [25664/60000]
loss: 2.020505 [32064/60000]
loss: 2.003184 [38464/60000]
loss: 1.945667 [44864/60000]
loss: 1.886640 [51264/60000]
loss: 1.921389 [57664/60000]
Test Error: 
 Accuracy:  55.3%, Avg loss: 1.861496 

Epoch 3
--------------------------------------
loss: 1.860651 [   64/60000]
loss: 1.909535 [ 6464/60000]
loss: 1.801010 [12864/60000]
loss: 1.754059 [19264/60000]
loss: 1.744402 [25664/60000]
loss: 1.656005 [32064/60000]